In [83]:
import argparse
import os

import pandas as pd
import pickle
from tqdm import tqdm
from copy import deepcopy

import numpy as np
import torch
from torch import optim
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt

import sys

In [84]:
from data_utils import load_data_all

dataset = 'gene'
D = np.load('/home2/zzs/dataset/gene/all22.npy')
M = D
d1, d2 = M.shape
print(M.shape)
r = 5

"""
dataset = 'netflix'
D = torch.load('/home2/zzs/dataset/netflix/netflix/use_data.pt')
D = D.numpy()
#s_r = 10000
#M=D[:s_r]
M = D
d1, d2 = M.shape
print(M.shape)
r = 5
"""
"""
dataset = 'ml-1m'
D = load_data_all(dataset, s=200)

print(f"original user: {D.shape[0]} item: {D.shape[1]}")
D = D.numpy()
r = 5
"""
"""
dataset = './data/syn/15000.npy'
M = np.load(dataset)
D = M
d1, d2 = M.shape
r = 5
"""


(1103547, 20)


"\ndataset = './data/syn/15000.npy'\nM = np.load(dataset)\nD = M\nd1, d2 = M.shape\nr = 5\n"

In [85]:
def differential_private_initialization(Y, G, tau, sigma0, p):
    m, n = Y.shape

    def projection(V, alpha2):
        V_projected = np.copy(V)
        for i in range(V_projected.shape[0]):
            row_norm = np.linalg.norm(V_projected[i, :], 2)
            if row_norm > alpha2:
                V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
        return V_projected

    # Step 1: Projection
    Y = projection(Y, G)

    # Step 2: Communicate (receive)
    Y_i = Y  # All users send their data to the server

    # Step 4: Server calculates private covariance matrix A
    A = tau**2 * np.sum([np.outer(Y_i[i], Y_i[i]) for i in range(m)], axis=0) / p**2 + np.random.normal(0, sigma0, size=(n, n))
    #A = (A + A.T) / 2  # Ensure A is symmetric

    # Step 5: Obtain (V^0, Σ) by performing the rank r SVD of A
    _, S, Vt = np.linalg.svd(A, full_matrices=False)

    V0 = Vt[:r, :].T
    Sigma = np.diag(S[:r])
    V0_tilde = V0 * np.sqrt(S[:r])

    U0 = np.array([(tau * Y[i].T @ V0 * np.sqrt(S[:r])/p) for i in range(m)])

    return U0.T, V0_tilde

# 初始化参数



#U0, V0_tilde = differential_private_initialization(Y, G, tau, sigma0, p)
#print("Initialized U0^T:\n", U0)
#U0 = U0.T

In [86]:
import numpy as np

p = 0.75

m, n = D.shape
runs = 5
ret_list = []
XTX_list = []
for run in range(runs):
    # 示例初始化
    m, n = D.shape
    Omega = np.random.rand(m, n) <= p 

    Omega = Omega * (D>0)
    non_zero_rows = np.any(D * Omega != 0, axis=1)
    M = D[non_zero_rows]
    Omega = Omega[non_zero_rows]
    m, n = M.shape
    mask = Omega
    Y = M * mask   # 示例数据
    """
    M = D
    m, n = M.shape
    Omega = np.random.rand(m, n) <= p 
    mask = Omega
    Y = M * mask
    """

    def P_omega(X):
        return X * mask

    def P_omegai(X, i):
        return X * mask[i]

    def differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2):
        """
        Ut -> U
        """

        U = U0
        V = V0_tilde
        Y_hat = np.array([V @ U[i] for i in range(m)])

        
        def projection(X, alpha):
            norm = np.linalg.norm(X, ord=2, axis=None)
            return X if norm <= alpha else (alpha / norm) * X
        
        def projection_(V, alpha):

            V_projected = np.copy(V)
            for i in range(V_projected.shape[0]):
                row_norm = np.linalg.norm(V_projected[i, :], 2)
                if row_norm > alpha2:
                    V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
            return V_projected
        
        def projection_1d(V, alpha2):

            row_norm = np.linalg.norm(V, 2)
            if row_norm > alpha2:
                V = (alpha2 / row_norm) * V
            return V

        for t in tqdm(range(T)):
            if t == 0:            
                # 用户端接收
                P_omega_Y_hat_minus_Y = P_omega(Y_hat - Y)

            # 服务器端计算
            P_omega_Y_hat_minus_Y = projection(P_omega_Y_hat_minus_Y, G)

            R_tilde = np.sum([U[i] @ U[i].T for i in range(m)], axis=0) - V.T  @ V + np.random.normal(0, sigma1, size=(r, r))
            V = V - (eta / p) * (P_omega_Y_hat_minus_Y.T @ U + np.random.normal(0, sigma2, size=(n, r))) + (eta / 2 ) * V @ R_tilde
            V = projection(V, alpha2)

            # 用户端更新
            for i in range(m):
                U[i] = U[i] - (eta / p) * P_omegai(Y_hat[i].T - Y[i].T, i) @ V - (eta / 2) * U[i].T @ R_tilde
                U[i] = projection_1d(U[i], alpha1)
                Y_hat[i] = V @ U[i].T


        return U, V

    # 初始化参数
    print(M.shape)
    T = 10
    alpha1 = 3
    alpha2 = 3
    G = alpha1 * alpha2
    eta = 0.3
    sigma1 = 0.0
    sigma2 = 0.0
    tau = 0.1
    sigma0 = 0.0
    U0, V0_tilde = differential_private_initialization(Y, G, tau, sigma0, p)
    U0 = U0.T


    print(U0.shape)
    print(V0_tilde.shape)

    #print("Initialized U0^T:\n", U0)

    U_final, V_final = differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2)

    X_estimation = U_final @ V_final.T
    mask_test = mask * (M>0.1)
    mask_test = mask
    #print(np.sum(mask_test))
    rand_mat = np.random.rand(m,n)
    #print((np.linalg.norm(P_omega(M))**2)/(m*n))
    #print((np.linalg.norm((M - rand_mat)*mask_test)**2)/np.sum(mask_test))
    #rmse = np.sqrt(np.sum(P_omega(M - X_estimation)**2/np.sum(mask)))
    #rmse = (np.linalg.norm(P_omega(M - X_estimation))**2)/(np.sum(mask))
    rmse = (np.linalg.norm(M - X_estimation)**2)/(m*n)
    rmse2 = (np.linalg.norm((M - X_estimation)*mask_test)**2)/np.sum(mask_test)
    ret_list.append(rmse2)
    #print(rmse)
    #print(np.sqrt(rmse2))

    MTM = M.T @ M
    XTX = X_estimation.T @ X_estimation
    err = np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM)
    #print(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))
    XTX_list.append(err)
    print("end iter")
#print("Final U:\n", U_final)
#print("Final V:\n", V_final)
print(np.mean(ret_list))
print(np.std(ret_list))
print(np.mean(XTX_list))
print(np.std(XTX_list))

(1103523, 20)
(1103523, 5)
(20, 5)


100%|██████████| 10/10 [05:09<00:00, 30.96s/it]


end iter
(1103521, 20)
(1103521, 5)
(20, 5)


100%|██████████| 10/10 [05:10<00:00, 31.03s/it]


end iter
(1103522, 20)
(1103522, 5)
(20, 5)


...

end iter
(1103521, 20)
(1103521, 5)
(20, 5)


100%|██████████| 10/10 [05:05<00:00, 30.58s/it]


end iter
(1103519, 20)
(1103519, 5)
(20, 5)


100%|██████████| 10/10 [05:06<00:00, 30.63s/it]


end iter
1.1301229060473323
0.00035603437846172173
62208.90712403358
2.657294593268179


In [87]:
MTM = M.T @ M
VTV = V_final @ V_final.T
print(np.linalg.norm(VTV-MTM) / np.linalg.norm(MTM))

0.9990730197639328


In [88]:
XTX = X_estimation.T @ X_estimation
print(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))

62207.39735138305
